In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType,FloatType

import pyspark.sql.functions as F

catalog_name="ecommerce"

### **BRANDS**

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)

In [0]:
#trim brand name
df_silver= df_bronze.withColumn("brand_name",F.trim(F.col("brand_name")))
df_silver.show(10)

In [0]:
#replace irregular characters
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))
df_silver.show(10)
                                 

In [0]:
df_silver.select("brand_category").distinct().show()

In [0]:
#replace anomalies
anomalies= {
     "BOOKS":"BKS",
     "GROCERY":"GRCY",
     "TOYS":"TOY"
 }

df_silver = df_silver.replace(anomalies, subset="brand_category")
df_silver.select("brand_category").distinct().show()

In [0]:
#write data to silver layer(catelog: ecommerce schema:silver table: slv_brands)
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

### **CATEGORY**

In [0]:
df_category = spark.table(f"{catalog_name}.bronze.brz_category")
df_category.show(10)


In [0]:
#look for duplicates
df_duplicates =df_category.groupby("category_code").count().filter("count > 1")
df_duplicates.show()


In [0]:
#drop duplicates
df_category = df_category.dropDuplicates(["category_code"])
df_category.show(10)

In [0]:
#category_code to Upper
df_category = df_category.withColumn("category_code", F.upper(F.col("category_code")))
df_category.show(10)


In [0]:
#write to delta
df_category.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_category")

### **PRODUCTS**

In [0]:
#read products table
df_products = spark.table(f"{catalog_name}.bronze.brz_products")
df_products.show(10)

In [0]:
rowcount,columncount= df_products.count(),len(df_products.columns)

print(f"rowcount:{rowcount}")
print(f"rowcount:{columncount}")

In [0]:
#check distinct material values
df_products.select("material").distinct().show()

In [0]:
#correcting material spellings
df_spelling =df_products.withColumn(
    "material", 
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Ruber", "Rubber")
     .when(F.col("material") == "Alumium", "Aluminum")
     .otherwise(F.col("material"))
)
df_spelling.select("material").distinct().show()


In [0]:
#removing g in weight_grams
df_dropgrams = df_spelling.withColumn(
    "weight_grams", 
    F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType())
)
df_dropgrams.select("weight_grams").show()

In [0]:
df_check=df_dropgrams.select("length_cm","width_cm","height_cm").show(10)

In [0]:
from pyspark.sql.functions import cast

In [0]:
#transform length_cm
df_dropLgrams = df_dropgrams.withColumn(
    "length_cm", 
    F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType())
)
df_dropLgrams.select("length_cm").show()

In [0]:
#category and brand code to upper
df_toUpper = df_dropLgrams.withColumn(
    "category_code", 
    F.upper(F.col("category_code"))
).withColumn(
    "brand_code", 
    F.upper(F.col("brand_code"))
    )
df_toUpper.show(10)

In [0]:
#show rating count
df_toUpper.select("rating_count").filter(F.col("rating_count")<0).show(10)


In [0]:
#convert negative rating_count to positive
df_toUpper = df_toUpper.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
    .otherwise(F.lit(0)) #if null the rating_count is 0
)

df_toUpper.select("rating_count").filter(F.col("rating_count")<0).show(10)

In [0]:
#check final cleaned data
df_final_cleaned_data= df_toUpper.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count")

df_final_cleaned_data.show(10)


In [0]:
df_full_data=df_toUpper.show(10)

In [0]:
#write final_cleaned_data to delta
df_toUpper.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_products")

### **CUSTOMERS**

In [0]:
#read customers table
df_customers=spark.read.table(f"{catalog_name}.bronze.brz_customers")
df_customers.show(10)

In [0]:
row_count,column_count=df_customers.count(),len(df_customers.columns)
print(f"row_count:{row_count}, column_count:{column_count}")

In [0]:
null_count=df_customers.filter(F.col("customer_id").isNull()).count()
print(f"null_count:{null_count}")


In [0]:
drop_nulls_ids=df_customers.dropna(subset=["customer_id"])
check_df=drop_nulls_ids.select("customer_id").count()
print(check_df)

In [0]:
df_null_phones=drop_nulls_ids.filter(F.col("phone").isNull()).count()
print(df_null_phones)

In [0]:
#replace null phones with "not available"
df_replace_null_phones=drop_nulls_ids.withColumn(
    "phone",
    F.when(F.col("phone").isNull(), F.lit("not available"))
    .otherwise(F.col("phone"))
)
df_replace_null_phones.show(10)

In [0]:
#write customers to delta
df_replace_null_phones.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_customers")


### **CALENDER**/**DATE**

In [0]:
#read date table
df_date=spark.read.table(f"{catalog_name}.bronze.brz_date")
df_date.show(10)



In [0]:
df_date.printSchema()

In [0]:
from pyspark.sql.functions import to_timestamp, to_date, col

df_date = df_date.withColumn(
    "date",
    to_date(
        to_timestamp(col("date"), "dd-MM-yyyy")
    )
)

df_date.show(10)


In [0]:
duplicates=df_date.groupby("date").count().filter(("count > 1"))
print(duplicates.count())
display(duplicates)


In [0]:
#count date rows
date_row_count = df_date.count()
print(f"date_row_count: {date_row_count}")

In [0]:
#drop duplicates
df_date=df_date.dropDuplicates(["date"])

count_after=df_date.count()
print(f"date_row_count: {count_after}")

In [0]:
#normalize day_names
df_date=df_date.withColumn("day_name", F.initcap(F.col("day_name")))
df_date.show(10)

In [0]:
#convert negative week_of_year to positive
df_date=df_date.withColumn("week_of_year", F.abs(F.col("week_of_year")).cast("int"))
df_date.show(10)


In [0]:
#Enhance quarter and week_of_year column

df_date=df_date.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"),F.col("quarter"), F.lit("-"),F.col("year"))))

df_date=df_date.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("week"), F.col("week_of_year"), F.lit("-"),F.col("year"))))

df_date.show(10)


In [0]:
df_date=df_date.withColumnRenamed("week_of_year", "week")
df_date.show(10)

In [0]:
#write data to delta

df_date.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_date")
